# 05 Gate separability (E4b)

Answers one question: when the Mahalanobis gate fails to withhold a held-out subtype,
is that because the score carries no signal, or because the 95th-percentile threshold
sits in the wrong place?

It replays the E2 hold-out loop with the same seeds and the same splits, using the
`NoveltyGate` from `sih_model.py`, so the withheld fractions it reproduces must match
`reports/v4/e2_leave_one_subtype_out.csv`. Cell 4 checks that and stops if they differ.

Run all. Nothing in `src/sih_model.py` is modified.

Requires `src/sih_gate_auroc.py`.


In [ ]:

# 1. setup
import sys, subprocess, pathlib, warnings
warnings.filterwarnings("ignore")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

subprocess.run(["pip","install","-q","xgboost","scikit-learn","scipy","pandas","matplotlib"], check=True)

REPO = pathlib.Path("/content/drive/MyDrive/UAV_GNSS_Research/uav-gnss-triage")
sys.path.insert(0, str(REPO / "src"))
import paths as P

import numpy as np, pandas as pd
import sih_model as M
import sih_gate_auroc as G

# ---- config -------------------------------------------------------------
FEATURE_RUN = "features_v3"
OUT         = P.REPORTS / "v4"
SEEDS       = [0, 1, 2]     # must match --e2_reps used for reports/v4
SUBTYPES    = None          # None = all eight. Set e.g. ["baro_stuck"] for a smoke test.
N_PERM      = 200           # permutations for the best-feature null
PCA_K       = 10
# -------------------------------------------------------------------------

assert (REPO / "src" / "sih_gate_auroc.py").exists(), "put sih_gate_auroc.py in src/ first"
print("repo   :", REPO)
print("out    :", OUT)
print("seeds  :", SEEDS)

In [ ]:

# 2. load the same windows and flight index sih_model.py uses
W  = M.engineer(pd.read_csv(P.FEATURES / FEATURE_RUN / "windows.csv"))
feats = [c for c in W.columns
         if c not in M.META and not c.startswith("rx_") and pd.api.types.is_numeric_dtype(W[c])]
Ws = W[W.source == "sih"].reset_index(drop=True)
F  = Ws.groupby("flight_id").agg(family=("family","first"), subtype=("subtype","first")).reset_index()

print(f"features used: {len(feats)}")
print("SIH windows:", len(Ws), "| flights:", len(F))
print("subtypes:", sorted(F.subtype.unique()))

In [ ]:

# 3. replay the E2 loop and score the gate
#    The split code below is copied verbatim from sih_model.py main(), including the
#    order of the CLASSES loop, so RandomState(700 + seed) yields identical splits.
import time
rows, t0 = [], time.time()

targets = []
for fam in M.CLASSES[1:]:
    for st in sorted(F.loc[F.family == fam, "subtype"].unique()):
        if SUBTYPES is None or st in SUBTYPES:
            targets.append((fam, st))
print(f"{len(targets)} hold-outs x {len(SEEDS)} seeds = {len(targets)*len(SEEDS)} pipeline fits\n")

for fam, st in targets:
    held = F.loc[F.subtype == st, "flight_id"].tolist()
    for seed in SEEDS:
        rng  = np.random.RandomState(700 + seed)
        seen = F.loc[(F.family == fam) & (F.subtype != st), "flight_id"].tolist(); rng.shuffle(seen)
        n_tr = len(seen) // 3; n_ca = len(seen) // 3
        tr, ca = seen[:n_tr], seen[n_tr:n_tr + n_ca]; seen_test = seen[n_tr + n_ca:]
        for c in M.CLASSES:
            if c == fam:
                continue
            ids = F.loc[F.family == c, "flight_id"].tolist(); rng.shuffle(ids)
            tr += ids[:20]; ca += ids[20:40]

        W_tr, W_ca = M.sub(Ws, tr), M.sub(Ws, ca)
        wmodel, fmodel, agg_cols, T = M.stacked_pipeline(W_tr, feats, 700 + seed)

        A_tr   = M.flight_aggregates(W_tr, M.stacked_pipeline.last_oof)
        A_ca   = M.flight_aggregates(W_ca, M.predict_proba(wmodel, W_ca[feats]))
        A_seen = M.flight_aggregates(M.sub(Ws, seen_test), M.predict_proba(wmodel, M.sub(Ws, seen_test)[feats]))
        A_held = M.flight_aggregates(M.sub(Ws, held),      M.predict_proba(wmodel, M.sub(Ws, held)[feats]))

        gate = M.NoveltyGate(q=0.05, seed=700 + seed).fit(A_tr, agg_cols).calibrate(A_ca)
        s_ca, s_seen, s_held = gate.scores(A_ca), gate.scores(A_seen), gate.scores(A_held)

        S_seen = M.stacked_scores(M.sub(Ws, seen_test), M.predict_proba(wmodel, M.sub(Ws, seen_test)[feats]), fmodel, agg_cols)
        S_held = M.stacked_scores(M.sub(Ws, held),      M.predict_proba(wmodel, M.sub(Ws, held)[feats]),      fmodel, agg_cols)

        # PCA sensitivity: is a low AUROC an estimation problem or an absent signal?
        Xtr, (Xun, Xse), _ = G._standardise(
            A_tr[agg_cols].to_numpy(float),
            [A_held[agg_cols].to_numpy(float), A_seen[agg_cols].to_numpy(float)])
        _, Xca_l, _ = G._standardise(A_tr[agg_cols].to_numpy(float), [A_ca[agg_cols].to_numpy(float)])
        pca_out = G._mahalanobis_scores(Xtr, [Xca_l[0], Xun, Xse], pca_k=PCA_K, seed=700 + seed)

        scorers = {
            "mahalanobis":       (s_ca["mahal"], s_held["mahal"], s_seen["mahal"]),
            "iforest":           (s_ca["iso"],   s_held["iso"],   s_seen["iso"]),
            "one_minus_conf":    (np.array([]),  1 - S_held.conf.to_numpy(float), 1 - S_seen.conf.to_numpy(float)),
        }
        if pca_out is not None:
            scorers[f"mahalanobis_pca{PCA_K}"] = tuple(pca_out)

        feat_tbl = G.per_feature_auroc(A_held[agg_cols].to_numpy(float),
                                       A_seen[agg_cols].to_numpy(float),
                                       feature_names=list(agg_cols), top=1)
        null = G.best_feature_null(A_held[agg_cols].to_numpy(float),
                                   A_seen[agg_cols].to_numpy(float),
                                   n_perm=N_PERM, seed=700 + seed)

        for name, (c_, u_, se_) in scorers.items():
            r = {"held_out_family": fam, "held_out_subtype": st, "seed": seed, "score": name,
                 "n_train": len(A_tr), "n_calib": len(c_), "n_agg_features": len(agg_cols),
                 "best_feature": (str(feat_tbl.iloc[0]["feature"]) if len(feat_tbl) else None),
                 "best_feature_auroc": (float(feat_tbl.iloc[0]["auroc_abs"]) if len(feat_tbl) else np.nan),
                 "best_feature_null_q95": null.get("best_feature_null_q95", np.nan),
                 "best_feature_perm_p":   null.get("best_feature_perm_p",   np.nan)}
            r.update(G.auroc_with_ci(u_, se_))
            r.update(G.tpr_at_fixed_fpr(u_, se_, fpr=0.05))
            if len(c_):
                thr = float(np.quantile(c_, 0.95))
                r["threshold_q95"]       = thr
                r["withheld_unseen_q95"] = float(np.mean(np.asarray(u_)  > thr))
                r["withheld_seen_q95"]   = float(np.mean(np.asarray(se_) > thr))
            else:
                r["threshold_q95"] = r["withheld_unseen_q95"] = r["withheld_seen_q95"] = np.nan
            rows.append(r)

        print(f"  {st:18s} seed {seed}  "
              f"AUROC(mahal)={[x for x in rows if x['score']=='mahalanobis'][-1]['auroc']:.3f}  "
              f"[{time.time()-t0:5.0f}s]")

PS = pd.DataFrame(rows)
PS.to_csv(OUT / "e4b_gate_separability_per_seed.csv", index=False)
print(f"\ndone in {time.time()-t0:.0f}s, {len(PS)} rows")

In [ ]:

# 4. reproduction check: the withheld fractions must match the committed E2 table
loso = pd.read_csv(OUT / "e2_leave_one_subtype_out.csv")
chk  = (PS[PS.score.isin(["mahalanobis","iforest"])]
        .groupby(["held_out_subtype","score"])[["withheld_unseen_q95","withheld_seen_q95"]]
        .mean().reset_index())

ref = loso.set_index("held_out_subtype")
bad, checked = [], 0
for _, r in chk.iterrows():
    key = "mahal" if r["score"] == "mahalanobis" else "iso"
    if r["held_out_subtype"] not in ref.index:
        continue
    for side, col in (("unseen", f"unseen_gate_{key}_withheld_mean"),
                      ("seen",   f"seen_gate_{key}_withheld_mean")):
        if col not in ref.columns:
            continue
        got, want = r[f"withheld_{side}_q95"], float(ref.loc[r["held_out_subtype"], col])
        checked += 1
        if abs(got - want) > 0.02:
            bad.append((r["held_out_subtype"], r["score"], side, round(got,3), round(want,3)))

if bad:
    print("MISMATCH against reports/v4 (subtype, score, side, replayed, committed):")
    for b in bad: print("  ", b)
    raise SystemExit("replay does not reproduce the committed E2 gate numbers; do not use this table")
print(f"reproduction check passed on {checked} comparisons (tolerance 0.02)")

In [ ]:

# 5. build the table
T = G.summarise_seeds(PS, by=("held_out_family","held_out_subtype","score"))
cols = ["held_out_family","held_out_subtype","score",
        "auroc_mean","auroc_std","auroc_lo_mean","auroc_hi_mean","mw_p_mean",
        "tpr_at_fpr_mean","withheld_unseen_q95_mean","withheld_seen_q95_mean",
        "best_feature_auroc_mean","best_feature_null_q95_mean","best_feature_perm_p_mean",
        "n_pos_mean","n_neg_mean"]
cols = [c for c in cols if c in T.columns]
T = T[cols]

(OUT / "tables").mkdir(parents=True, exist_ok=True)
T.round(4).to_csv(OUT / "tables" / "t6e_gate_separability.csv", index=False)

pd.set_option("display.width", 250)
print("=== Mahalanobis gate, per hold-out ===")
print(T[T.score == "mahalanobis"][
    ["held_out_subtype","auroc_mean","auroc_lo_mean","auroc_hi_mean",
     "tpr_at_fpr_mean","withheld_unseen_q95_mean","withheld_seen_q95_mean"]].round(3).to_string(index=False))
print()
print("=== all scores ===")
print(T.round(3).to_string(index=False))

In [ ]:

# 6. verdict per hold-out, using the rules in the module docstring
v = T[T.score == "mahalanobis"].set_index("held_out_subtype")
best = T[T.score == "mahalanobis"].set_index("held_out_subtype")

def verdict(st):
    a, lo, hi = v.loc[st,"auroc_mean"], v.loc[st,"auroc_lo_mean"], v.loc[st,"auroc_hi_mean"]
    tpr, wq   = v.loc[st,"tpr_at_fpr_mean"], v.loc[st,"withheld_unseen_q95_mean"]
    bf, bnull = best.loc[st,"best_feature_auroc_mean"], best.loc[st,"best_feature_null_q95_mean"]
    p         = best.loc[st,"best_feature_perm_p_mean"]
    if a >= 0.80 and wq < 0.50:
        return "SEPARABLE, threshold wrong -> restate the operating point"
    if a >= 0.80:
        return "separable and caught -> gate works here"
    if bf > bnull and p < 0.05:
        return "multivariate distance buries a real univariate signal -> estimation problem"
    if lo <= 0.5 <= hi:
        return "NO SIGNAL in the aggregates -> no threshold rescues the gate"
    return "weak, inconclusive at this sample size"

print(f"{'subtype':20s} {'AUROC':>7s} {'TPR@5%FPR':>10s} {'withheld':>9s}   verdict")
for st in v.index:
    print(f"{st:20s} {v.loc[st,'auroc_mean']:7.3f} {v.loc[st,'tpr_at_fpr_mean']:10.3f} "
          f"{v.loc[st,'withheld_unseen_q95_mean']:9.3f}   {verdict(st)}")

In [ ]:

# 7. figure
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = P.FIGURES / "v4"; FIG.mkdir(parents=True, exist_ok=True)
m = T[T.score == "mahalanobis"].sort_values("auroc_mean")
y = np.arange(len(m))

fig, ax = plt.subplots(1, 2, figsize=(11, 0.55*len(m)+2.4), sharey=True)
ax[0].errorbar(m.auroc_mean, y,
               xerr=[m.auroc_mean-m.auroc_lo_mean, m.auroc_hi_mean-m.auroc_mean],
               fmt="o", color="#1f3864", capsize=3)
ax[0].axvline(0.5, color="grey", ls="--", lw=1)
ax[0].set_xlim(0, 1.02); ax[0].set_xlabel("AUROC, held-out vs seen-subtype test")
ax[0].set_yticks(y); ax[0].set_yticklabels(m.held_out_subtype)
ax[0].set_title("Separability of the Mahalanobis score")

w = 0.38
ax[1].barh(y - w/2, m.tpr_at_fpr_mean,          height=w, label="caught at 5% false abstention", color="#2e5597")
ax[1].barh(y + w/2, m.withheld_unseen_q95_mean, height=w, label="withheld at the 95th percentile", color="#c0c0c0")
ax[1].set_xlim(0, 1.02); ax[1].set_xlabel("fraction of held-out flights")
ax[1].legend(loc="lower right", fontsize=8)
ax[1].set_title("Best available cut vs the cut in use")

fig.tight_layout()
for ext in ("png","pdf"):
    fig.savefig(FIG / f"f9_gate_separability.{ext}", dpi=200, bbox_inches="tight")
print("wrote", FIG / "f9_gate_separability.png")
plt.show()

In [ ]:
# 8. commit
import os, subprocess
os.chdir(str(REPO))
subprocess.run(["pip", "install", "-q", "nbconvert", "jupyter"], check=False)
r = subprocess.run(["python", "tools/commit_cell.py",
                    "E4b gate separability: AUROC, TPR at 5% false abstention, per-feature null"],
                   capture_output=True, text=True)
print(r.stdout, r.stderr)